In [1]:
from drive_utils import read_csv_from_drive, get_file_id_from_url


# 방법 2: 공유 링크에서 파일 ID 추출
drive_url = "https://drive.google.com/file/d/19rV5-AOopv859LwedTZhJyiUajFUMm7e/view?usp=drive_link"
file_id = get_file_id_from_url(drive_url)
df = read_csv_from_drive(file_id)

print(df.head())
print(f"\n컬럼 목록: {list(df.columns)}")


🔄 토큰 갱신 중...
✅ 인증 완료!
📖 파일 읽는 중... (메모리 로드, 저장 안함)
✅ 조회 완료! 데이터 크기: 546,028행 x 16열
   subject_id   hadm_id            admittime            dischtime deathtime  \
0    10000032  22595853  2180-05-06 22:23:00  2180-05-07 17:15:00       NaN   
1    10000032  22841357  2180-06-26 18:27:00  2180-06-27 18:49:00       NaN   
2    10000032  25742920  2180-08-05 23:44:00  2180-08-07 17:50:00       NaN   
3    10000032  29079034  2180-07-23 12:35:00  2180-07-25 17:55:00       NaN   
4    10000068  25022803  2160-03-03 23:16:00  2160-03-04 06:26:00       NaN   

   admission_type admit_provider_id      admission_location  \
0          URGENT            P49AFC  TRANSFER FROM HOSPITAL   
1        EW EMER.            P784FA          EMERGENCY ROOM   
2        EW EMER.            P19UTS          EMERGENCY ROOM   
3        EW EMER.            P06OTX          EMERGENCY ROOM   
4  EU OBSERVATION            P39NWO          EMERGENCY ROOM   

  discharge_location insurance language marital_status   race  

In [5]:

# 방법 3: 특정 컬럼만 읽기 (메모리 절약)
df = read_csv_from_drive(
    file_id, 
    usecols=['subject_id', 'hadm_id', 'admittime', 'dischtime']
)

print(df.info())

# 방법 4: 데이터 분석
print(f"\n총 환자 수: {df['subject_id'].nunique()}")
print(f"\n기본 통계:\n{df.describe()}")


📥 파일 다운로드 중...
✅ 성공! 데이터 크기: 546028행 x 4열
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   subject_id  546028 non-null  int64 
 1   hadm_id     546028 non-null  int64 
 2   admittime   546028 non-null  object
 3   dischtime   546028 non-null  object
dtypes: int64(2), object(2)
memory usage: 16.7+ MB
None

총 환자 수: 223452

기본 통계:
         subject_id       hadm_id
count  5.460280e+05  5.460280e+05
mean   1.501118e+07  2.500100e+07
std    2.877694e+06  2.888710e+06
min    1.000003e+07  2.000002e+07
25%    1.252380e+07  2.249662e+07
50%    1.501961e+07  2.500385e+07
75%    1.750403e+07  2.750282e+07
max    1.999999e+07  2.999994e+07


In [ ]:
from drive_utils import read_csv_from_drive, get_file_id_from_url

# 1. Admissions 파일 로드
adm_url = "https://drive.google.com/file/d/19rV5-AOopv859LwedTZhJyiUajFUMm7e/view?usp=drive_link"
adm_file_id = get_file_id_from_url(adm_url)
df_adm = read_csv_from_drive(adm_file_id)

# 2. Diagnoses ICD 파일 로드
diag_url = "https://drive.google.com/file/d/1FWWjGHLMJnqfKcJeiGFTMEGOpBin8Skq/view?usp=drive_link"
diag_file_id = get_file_id_from_url(diag_url)
df_diag = read_csv_from_drive(diag_file_id)

# 3. 컬럼 확인 (디버깅용)
print("=== Admissions 컬럼 ===")
print(list(df_adm.columns))
print("\n=== Diagnoses 컬럼 ===")
print(list(df_diag.columns))

# 4. 사망환자 필터링 (hospital_expire_flag == 1)
df_deceased = df_adm[df_adm['hospital_expire_flag'] == 1]
print(f"\n총 사망 환자 수: {len(df_deceased)}")

# 5. 사망환자의 진단 데이터만 추출 (hadm_id로 조인)
deceased_hadm_ids = df_deceased['hadm_id'].unique()
df_death_diag = df_diag[df_diag['hadm_id'].isin(deceased_hadm_ids)]

# 6. Primary diagnosis만 필터링 (seq_num == 1)
df_primary = df_death_diag[df_death_diag['seq_num'] == 1]

# 7. 사망원인 Top 5 추출
top5_causes = df_primary['icd_code'].value_counts().head(10)

print("\n=== 사망환자 사망원인 Top 10 (ICD 코드) ===")
print(top5_causes)

# 8. 결과를 DataFrame으로 변환
df_top5 = top5_causes.reset_index()
df_top5.columns = ['icd_code', 'count']
print("\n=== Top 10 DataFrame ===")
print(df_top5)


# d_icd_diagnoses 파일이 있다면 코드 설명 추가
d_icd_url = "https://drive.google.com/file/d/1qdp5z5qR847ZWp9sUYtzdwJY99VaWI7p/view?usp=drive_link"
d_icd_file_id = get_file_id_from_url(d_icd_url)
df_icd_desc = read_csv_from_drive(d_icd_file_id)
df_top5_with_desc = df_top5.merge(df_icd_desc[['icd_code', 'long_title']], on='icd_code', how='left')
print(df_top5_with_desc)

📖 파일 읽는 중... (메모리 로드, 저장 안함)
✅ 조회 완료! 데이터 크기: 546,028행 x 16열
📖 파일 읽는 중... (메모리 로드, 저장 안함)
✅ 조회 완료! 데이터 크기: 6,364,488행 x 5열
=== Admissions 컬럼 ===
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

=== Diagnoses 컬럼 ===
['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']

총 사망 환자 수: 11801

=== 사망환자 사망원인 Top 10 (ICD 코드) ===
icd_code
A419     834
0389     763
Z515     412
431      287
51881    191
A4189    188
U071     166
I130     146
486      114
I214     107
Name: count, dtype: int64

=== Top 10 DataFrame ===
  icd_code  count
0     A419    834
1     0389    763
2     Z515    412
3      431    287
4    51881    191
5    A4189    188
6     U071    166
7     I130    146
8      486    114
9     I214    107
📖 파일 읽는 중... (메모리 로드, 저장 안함)
✅ 조회 완료! 데이터 크기: 112,107행 x 3열
  icd_code  count 

In [ ]:
from drive_utils import read_csv_from_drive, get_file_id_from_url
import pandas as pd
import numpy as np

# ===== 파일 로드 =====
# 1. chartevents 파일 (Vital Signs)
chart_url = "Yhttps://drive.google.com/file/d/1FjViZIG3Fx7n9lvp_N0gdJNFLP1cBN1A/view?usp=drive_link"  # chartevents.csv.gz URL
chart_file_id = get_file_id_from_url(chart_url)
df_chart = read_csv_from_drive(chart_file_id)

# 2. labevents 파일 (Lab Results)
lab_url = "https://drive.google.com/file/d/1O1ZdjCPeEDE6O81svdzvsxN_FC2zb2sW/view?usp=drive_link"  # labevents.csv.gz URL
lab_file_id = get_file_id_from_url(lab_url)
df_lab = read_csv_from_drive(lab_file_id)

# 3. outputevents 파일 (Urine Output)
output_url = "https://drive.google.com/file/d/1rDtEbepXMSqX4D7KipbnIods8yAjS3Zt/view?usp=drive_link"  # outputevents.csv.gz URL
output_file_id = get_file_id_from_url(output_url)
df_output = read_csv_from_drive(output_file_id)

# 4. inputevents 파일 (IV Medications)
input_url = "https://drive.google.com/file/d/1u-fWUQ7eh4onHceyi_I6j8z9T1Mb2w39/view?usp=drive_link"  # inputevents.csv.gz URL
input_file_id = get_file_id_from_url(input_url)
df_input = read_csv_from_drive(input_file_id)

# 5. d_items 파일 (Item Dictionary)
items_url = "https://drive.google.com/file/d/10zRa12551b4Tem9Opa4yfh6AMBSi2t3K/view?usp=drive_link"  # d_items.csv.gz URL
items_file_id = get_file_id_from_url(items_url)
df_items = read_csv_from_drive(items_file_id)

# 6. icustays 파일 (ICU Stay Info - weight 포함)
icu_url = "https://drive.google.com/file/d/1nabPJ38RwwYGlVm_dEn_eat9dSmeDca4/view?usp=drive_link"  # icustays.csv.gz URL (선택사항)
icu_file_id = get_file_id_from_url(icu_url)
df_icu = read_csv_from_drive(icu_file_id)

print("=== 파일 로드 완료 ===")
print(f"chartevents: {len(df_chart)} rows")
print(f"labevents: {len(df_lab)} rows")
print(f"outputevents: {len(df_output)} rows")
print(f"inputevents: {len(df_input)} rows")

# ===== 추출 함수 =====
def extract_vitals(df_chart, stay_id):
    """MAP, SBP, SpO2, RR, FiO2 추출"""
    vitals = {}
    
    # MAP (Mean Arterial Pressure)
    map_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([220052, 220181, 225312]))
    ]
    vitals['MAP'] = map_data['valuenum'].median()
    
    # SBP (Systolic BP)
    sbp_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([220050, 220179]))
    ]
    vitals['SBP'] = sbp_data['valuenum'].median()
    
    # SpO2
    spo2_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([220277, 646]))
    ]
    vitals['SpO2'] = spo2_data['valuenum'].median()
    
    # Respiratory Rate
    rr_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([220210, 224690]))
    ]
    vitals['RR'] = rr_data['valuenum'].median()
    
    # FiO2 (0-1 scale)
    fio2_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([223835, 220292]))
    ]
    fio2_val = fio2_data['valuenum'].median()
    vitals['FiO2'] = fio2_val / 100 if fio2_val > 1 else fio2_val
    
    return vitals

def extract_lactate(df_lab, subject_id):
    """Lactate 추출"""
    lactate_data = df_lab[
        (df_lab['subject_id'] == subject_id) & 
        (df_lab['itemid'] == 50813)
    ]
    return lactate_data['valuenum'].median()

def extract_urine_output(df_output, stay_id, weight_kg):
    """Urine Output (mL/kg/hr)"""
    uo_data = df_output[
        (df_output['stay_id'] == stay_id) & 
        (df_output['itemid'].isin([226559, 226560, 226561, 226584]))
    ]
    total_uo = uo_data['value'].sum()
    
    # 시간 범위 계산
    if len(uo_data) > 0:
        hours = (pd.to_datetime(uo_data['charttime']).max() - 
                 pd.to_datetime(uo_data['charttime']).min()).total_seconds() / 3600
        if hours > 0 and weight_kg > 0:
            return total_uo / weight_kg / hours
    return np.nan

def extract_treatments(df_chart, df_input, stay_id):
    """산소투여, HFNC, 기계환기, 승압제"""
    treatments = {}
    
    # 산소 투여
    o2_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([223834, 227582]))
    ]
    treatments['oxygen'] = 'Yes' if len(o2_data) > 0 else 'No'
    
    # HFNC
    hfnc_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'] == 226732)
    ]
    treatments['HFNC'] = 'Yes' if len(hfnc_data) > 0 else 'No'
    
    # 기계환기
    mv_data = df_chart[
        (df_chart['stay_id'] == stay_id) & 
        (df_chart['itemid'].isin([720, 223849]))
    ]
    treatments['MechVent'] = 'Yes' if len(mv_data) > 0 else 'No'
    
    # 승압제
    vaso_data = df_input[
        (df_input['stay_id'] == stay_id) & 
        (df_input['itemid'].isin([221906, 221289, 221662]))
    ]
    treatments['Vasopressor'] = 'Yes' if len(vaso_data) > 0 else 'No'
    
    return treatments

def extract_patient_features(stay_id, subject_id, weight_kg=70):
    """모든 항목 통합 추출"""
    vitals = extract_vitals(df_chart, stay_id)
    lactate = extract_lactate(df_lab, subject_id)
    uo = extract_urine_output(df_output, stay_id, weight_kg)
    treatments = extract_treatments(df_chart, df_input, stay_id)
    
    result = {
        'Patient_ID': subject_id,
        'MAP': vitals.get('MAP', np.nan),
        'SBP': vitals.get('SBP', np.nan),
        'Lactate': lactate,
        'SpO2': vitals.get('SpO2', np.nan),
        'FiO2': vitals.get('FiO2', np.nan),
        'RR': vitals.get('RR', np.nan),
        'UO': uo,
        'Oxygen': treatments['oxygen'],
        'HFNC': treatments['HFNC'],
        'MechVent': treatments['MechVent'],
        'Vasopressor': treatments['Vasopressor']
    }
    
    return pd.DataFrame([result])

# ===== 사용 예시 =====
# 특정 환자 데이터 추출
stay_id = 30000194  # 예시 ICU stay_id
subject_id = 10000032  # 예시 subject_id
weight_kg = 70  # 환자 체중 (kg)

df_patient = extract_patient_features(stay_id, subject_id, weight_kg)
print("\n=== 환자 특징 추출 결과 ===")
print(df_patient)

# 여러 환자 배치 처리
stay_ids = df_icu['stay_id'].head(10).tolist()  # 예시: 10명
results = []
for sid in stay_ids:
    subj_id = df_icu[df_icu['stay_id'] == sid]['subject_id'].values[0]
    weight = 70  # 실제로는 df_icu에서 추출
    results.append(extract_patient_features(sid, subj_id, weight))

df_results = pd.concat(results, ignore_index=True)
print("\n=== 배치 추출 결과 ===")
print(df_results)


ParserError: Error tokenizing data. C error: Expected 1 fields in line 3, saw 4738
